# ShopFlow — Phase 2: STAGED Layer
### Transform RAW → STAGED · type cast · deduplicate · handle NULLs

---

### What this notebook does

Creates 9 tables in `SHOPFLOW_STAGED` — one per RAW table — with these transformations applied:

| Problem in RAW | Fix in STAGED |
|----------------|---------------|
| All columns are `VARCHAR` | Cast to `TIMESTAMP_NTZ`, `FLOAT`, `INTEGER` |
| Duplicate zip codes in geolocation | `ROW_NUMBER + QUALIFY` — keep 1 per zip |
| Column name typos in products | Rename with correct spelling |
| Portuguese category names | `LEFT JOIN` translation table — add English name |
| NULLs in key columns | Filter with `WHERE key IS NOT NULL` |

### Pattern used — CTAS (Create Table As Select)

Every STAGED table is built with:
```sql
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_x AS
SELECT ... FROM SHOPFLOW_RAW.RAW_x WHERE ...;
```
This is a complete rebuild every time. Fix a bug → re-run the cell → table is recreated cleanly from RAW. No partial updates, no state management.

### RAW stays untouched

STAGED reads from RAW but never modifies it. RAW is your permanent audit trail. If STAGED ever has a problem, you rebuild it from RAW without going back to Kaggle.

---

### Run cells one at a time — top to bottom

Each cell = one statement = one visible result.

---

### Key Snowflake concepts in this notebook

| Topic | Cells |
|-------|-------|
| `TRY_CAST` vs `CAST` | All transformation cells |
| `TRY_TO_TIMESTAMP` | STG_ORDERS, STG_REVIEWS |
| `CTAS` pattern | Every table cell |
| `ROW_NUMBER` + `QUALIFY` | STG_GEOLOCATION |
| `COALESCE` for NULL handling | STG_PRODUCTS, STG_REVIEWS |
| `LEFT JOIN` in CTAS | STG_PRODUCTS |
| `INFORMATION_SCHEMA.COLUMNS` | Verification cells |
| `DATEDIFF` | STG_ORDERS |
| `TIMESTAMP_NTZ` vs `TIMESTAMP_LTZ` | STG_ORDERS explanation |

---
## Cell 1 — Set context

Always set context explicitly at the top of every script. Never rely on whatever was set in a previous session.

**Why `USE SCHEMA SHOPFLOW_RAW`?**

All our source tables (`RAW_*`) live in `SHOPFLOW_RAW`. Setting it as the default schema means we can write `RAW_ORDERS` instead of `SHOPFLOW_RAW.RAW_ORDERS` in every `FROM` clause. Our `CREATE TABLE` statements will explicitly use `SHOPFLOW_STAGED.STG_*` to be unambiguous.

In [ ]:
--USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE  SHOPFLOW_DB;
USE SCHEMA    SHOPFLOW_RAW;

In [ ]:
-- Confirm context before doing anything
SELECT
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_DATABASE()  AS database,
    CURRENT_SCHEMA()    AS schema,
    CURRENT_ROLE()      AS role;

---
## Cell 2 — Preview RAW data before transforming

Before writing any transformation, always look at the actual data. This tells you:
- What the date format looks like (so you know which TRY_TO_TIMESTAMP format to use)
- Whether there are unexpected values in key columns
- How NULLs appear in the data

Run this cell to see 5 rows from each key RAW table.

In [ ]:
-- Preview RAW_ORDERS — check date format and NULL patterns
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,         -- e.g. '2018-01-08 21:28:27'
    order_delivered_customer_date,    -- NULL for undelivered orders
    order_estimated_delivery_date
FROM RAW_ORDERS
LIMIT 5;

select distinct order_status from raw_orders;

In [ ]:
-- Preview RAW_PRODUCTS — see the typos and Portuguese category names
SELECT
    product_id,
    product_category_name,            -- e.g. 'cama_mesa_banho' (Portuguese)
    product_name_lenght,              -- typo: should be product_name_length
    product_description_lenght,       -- typo: should be product_description_length
    product_weight_g
FROM RAW_PRODUCTS
LIMIT 5;

In [ ]:
-- Check duplicate zip codes in RAW_GEOLOCATION
-- This confirms why deduplication is needed
SELECT
    geolocation_zip_code_prefix,
    COUNT(*) AS row_count
FROM RAW_GEOLOCATION
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY 2 DESC
LIMIT 10;

---
## Cell 3 — Create STG_ORDERS

### What this table contains
One row per order. The central table of the entire dataset — every other table eventually joins back here via `order_id`.

### Transformations applied

**5 VARCHAR timestamps → TIMESTAMP_NTZ**

`TRY_TO_TIMESTAMP(order_purchase_timestamp)` converts the string `'2018-01-08 21:28:27'` to a proper timestamp. The `TRY_` prefix means malformed values return `NULL` instead of crashing.

**Why TIMESTAMP_NTZ?**

Snowflake has three timestamp types:
- `TIMESTAMP_NTZ` — No Time Zone. Stores value as-is. Best for DWH — store UTC, convert at reporting layer.
- `TIMESTAMP_LTZ` — Local Time Zone. Converts based on session timezone.
- `TIMESTAMP_TZ` — Timezone stored with the value.

Always use `TIMESTAMP_NTZ` for warehouse tables unless you explicitly need time zone conversions.

**Computed column: `delivery_delay_days`**

`DATEDIFF('day', estimated_delivery_ts, delivered_ts)` calculates how many days late (positive = late, negative = early, NULL = not yet delivered). This is a key analytics metric — computed once here, reused everywhere downstream.

**NULL filter: `WHERE order_id IS NOT NULL`**

An order without an `order_id` is meaningless — nothing can join to it. Filtering here means every row in STG_ORDERS is guaranteed to have a valid primary key.

> **💡 Concept Note: CTAS Pattern**
> 
> CTAS (`CREATE TABLE AS SELECT`) creates a table whose schema is inferred entirely from the SELECT statement. Column names come from aliases. Data types come from the expressions. `CREATE OR REPLACE` drops and recreates — makes the cell fully idempotent (safe to re-run).

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_ORDERS AS
SELECT
    order_id,
    customer_id,
    order_status,

    -- Cast all 5 VARCHAR timestamps to TIMESTAMP_NTZ
    -- TRY_TO_TIMESTAMP returns NULL for bad values instead of crashing
    TRY_TO_TIMESTAMP(order_purchase_timestamp)      AS order_purchase_ts,
    TRY_TO_TIMESTAMP(order_approved_at)             AS order_approved_ts,
    TRY_TO_TIMESTAMP(order_delivered_carrier_date)  AS carrier_pickup_ts,
    TRY_TO_TIMESTAMP(order_delivered_customer_date) AS delivered_ts,
    TRY_TO_TIMESTAMP(order_estimated_delivery_date) AS estimated_delivery_ts,

    -- Computed column: positive = delivered late, negative = delivered early
    -- NULL when order has not been delivered yet (delivered_ts is NULL)
    DATEDIFF(
        'day',
        TRY_TO_TIMESTAMP(order_estimated_delivery_date),
        TRY_TO_TIMESTAMP(order_delivered_customer_date)
    ) AS delivery_delay_days,

    -- Keep metadata columns for lineage tracing
    _loaded_at,
    _source_file

FROM RAW_ORDERS
WHERE order_id IS NOT NULL;  -- drop rows with no primary key — nothing can join to them

In [ ]:
-- Verify: check row count and sample the computed column
SELECT
    COUNT(*)                                    AS total_rows,
    COUNT(delivered_ts)                         AS delivered_orders,
    COUNT(*) - COUNT(delivered_ts)              AS not_yet_delivered,
    ROUND(AVG(delivery_delay_days), 1)          AS avg_delay_days,
    SUM(CASE WHEN delivery_delay_days > 0
             THEN 1 ELSE 0 END)                 AS late_orders,
    SUM(CASE WHEN delivery_delay_days <= 0
             THEN 1 ELSE 0 END)                 AS on_time_or_early
FROM SHOPFLOW_STAGED.STG_ORDERS;

---
## Cell 4 — Create STG_ORDER_ITEMS

### What this table contains
One row per item within an order. More rows than STG_ORDERS because one order can have multiple items. The combination of `order_id + order_item_id` is the unique key.

### Transformations applied

**`price` and `freight_value` → FLOAT**

These are the core financial measures of the dataset. Cast from VARCHAR to FLOAT using `TRY_CAST`. In STAGED they become real numbers — ready for `SUM()`, `AVG()`, `MAX()` without explicit casting in every query.

**`order_item_id` → INTEGER**

Position of the item within the order (1, 2, 3...). Stored as VARCHAR in RAW. Cast to INTEGER for correct ordering and arithmetic.

**`shipping_limit_date` → TIMESTAMP_NTZ**

Deadline for the seller to hand the package to the carrier. Useful for SLA analysis.

> **🛡️ Reliability Tip: TRY_CAST vs CAST**
> 
> `TRY_CAST(col AS FLOAT)` vs `CAST(col AS FLOAT)`: if `col` contains `'R$29,90'` or any non-numeric string — `CAST` throws an error and the query fails. `TRY_CAST` returns NULL and the query succeeds. In a RAW → STAGED pipeline, always use `TRY_CAST` — you cannot know in advance what dirty values exist in the source.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_ORDER_ITEMS AS
SELECT
    order_id,
    TRY_CAST(order_item_id AS INTEGER)       AS order_item_id,  -- position within order: 1,2,3...
    product_id,
    seller_id,
    TRY_TO_TIMESTAMP(shipping_limit_date)    AS shipping_limit_ts,
    TRY_CAST(price AS FLOAT)                 AS price,          -- item price in BRL
    TRY_CAST(freight_value AS FLOAT)         AS freight_value,  -- shipping cost in BRL
    _loaded_at,
    _source_file
FROM RAW_ORDER_ITEMS
WHERE order_id IS NOT NULL
  AND product_id IS NOT NULL
  AND seller_id IS NOT NULL;

In [ ]:
-- Verify: confirm numeric columns are now real numbers
SELECT
    COUNT(*)                       AS total_rows,
    ROUND(MIN(price), 2)           AS min_price_brl,
    ROUND(MAX(price), 2)           AS max_price_brl,
    ROUND(AVG(price), 2)           AS avg_price_brl,
    ROUND(SUM(price), 2)           AS total_gmv_brl,
    COUNT(CASE WHEN price IS NULL
               THEN 1 END)         AS null_prices
FROM SHOPFLOW_STAGED.STG_ORDER_ITEMS;

---
## Cell 5 — Create STG_CUSTOMERS

### What this table contains
Customer location data. ~99,441 rows — one per `customer_id` (which is per-order, not per person).

### The customer_id vs customer_unique_id distinction

This is the most important thing to understand about this table:

- `customer_id` — generated fresh for every order. One real person who places 5 orders appears 5 times with 5 different `customer_id` values.
- `customer_unique_id` — the true unique person identifier. Use this to count distinct customers.

```sql
-- WRONG — counts order-level IDs, not people
SELECT COUNT(DISTINCT customer_id) FROM STG_CUSTOMERS;

-- CORRECT — counts unique people
SELECT COUNT(DISTINCT customer_unique_id) FROM STG_CUSTOMERS;
```

### Transformations applied
Minimal changes — this table is mostly clean. `customer_zip_code_prefix` stays VARCHAR (it is a zip code, not a number — leading zeros matter, arithmetic is meaningless). State kept as VARCHAR 2-char code.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_CUSTOMERS AS
SELECT
    customer_id,
    customer_unique_id,                   -- TRUE unique person identifier
    customer_zip_code_prefix,             -- keep as VARCHAR — zip codes are not numbers
    customer_city,
    UPPER(customer_state) AS customer_state,  -- normalise to uppercase state code
    _loaded_at,
    _source_file
FROM RAW_CUSTOMERS
WHERE customer_id IS NOT NULL
  AND customer_unique_id IS NOT NULL;

In [ ]:
-- Verify: compare order-level vs person-level customer counts
-- The gap between the two shows how many customers placed multiple orders
SELECT
    COUNT(*)                          AS total_rows,
    COUNT(DISTINCT customer_id)       AS unique_order_ids,
    COUNT(DISTINCT customer_unique_id) AS unique_people,
    COUNT(DISTINCT customer_state)    AS states_covered
FROM SHOPFLOW_STAGED.STG_CUSTOMERS;

---
## Cell 6 — Create STG_SELLERS

### What this table contains
~3,095 rows — one per seller registered on the Olist platform. This becomes `DIM_SELLERS` in Phase 3.

### Transformations applied
Same pattern as customers — zip kept as VARCHAR, state normalised to uppercase. Clean and simple because seller data is already well-structured.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_SELLERS AS
SELECT
    seller_id,
    seller_zip_code_prefix,             -- keep as VARCHAR — zip codes are not numbers
    seller_city,
    UPPER(seller_state) AS seller_state, -- normalise to uppercase state code
    _loaded_at,
    _source_file
FROM RAW_SELLERS
WHERE seller_id IS NOT NULL;

In [ ]:
-- Verify: top 5 states by seller count
SELECT
    seller_state,
    COUNT(*) AS seller_count
FROM SHOPFLOW_STAGED.STG_SELLERS
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5;

---
## Cell 7 — Create STG_PRODUCTS

### What this table contains
~32,951 rows — one per product. The richest transformation cell in Phase 2.

### Transformations applied

**Fix column name typos**

The original Olist CSV has two typos: `product_name_lenght` and `product_description_lenght`. We preserved them in RAW (exact copy of source). In STAGED we rename:
```sql
product_name_lenght        AS product_name_length
product_description_lenght AS product_description_length
```
All downstream code references the correct spelling.

**Cast physical dimensions to FLOAT**

Weight (`product_weight_g`) and dimensions are used for freight cost modelling. They need to be real numbers. `TRY_CAST(... AS FLOAT)` handles any bad values.

**Add English category name via LEFT JOIN**

```sql
LEFT JOIN RAW_CATEGORY_TRANSLATION t
       ON p.product_category_name = t.product_category_name
```
Adds `product_category_name_english` directly to each product row. `COALESCE(t.product_category_name_english, 'uncategorised')` handles products with no translation — returns `'uncategorised'` instead of NULL.

**Why LEFT JOIN not INNER JOIN?**

If we used INNER JOIN, any product whose Portuguese category has no matching translation would be dropped entirely from the result. LEFT JOIN keeps all products — unmatched ones get NULL for the English name, which COALESCE converts to `'uncategorised'`. Never silently lose rows in a transformation.

> **💡 Concept Note: COALESCE**
> 
> `COALESCE(a, b, c)` returns the first non-NULL value in the list. It is standard SQL — not Snowflake-specific. Used constantly in STAGED layers for NULL replacement.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_PRODUCTS AS
SELECT
    p.product_id,
    p.product_category_name,

    -- Add English translation via LEFT JOIN
    -- COALESCE ensures no NULL — unmatched categories become 'uncategorised'
    COALESCE(
        t.product_category_name_english,
        'uncategorised'
    )                                                AS product_category_english,

    -- Fix typos from original dataset (lenght → length)
    TRY_CAST(p.product_name_lenght AS INTEGER)       AS product_name_length,
    TRY_CAST(p.product_description_lenght AS INTEGER) AS product_description_length,
    TRY_CAST(p.product_photos_qty AS INTEGER)        AS product_photos_qty,

    -- Cast physical dimensions to FLOAT for freight modelling
    TRY_CAST(p.product_weight_g AS FLOAT)            AS product_weight_g,
    TRY_CAST(p.product_length_cm AS FLOAT)           AS product_length_cm,
    TRY_CAST(p.product_height_cm AS FLOAT)           AS product_height_cm,
    TRY_CAST(p.product_width_cm AS FLOAT)            AS product_width_cm,

    p._loaded_at,
    p._source_file

FROM RAW_PRODUCTS p

-- LEFT JOIN — keeps all products even if no translation exists
-- INNER JOIN would silently drop products with missing translations
LEFT JOIN RAW_CATEGORY_TRANSLATION t
       ON p.product_category_name = t.product_category_name

WHERE p.product_id IS NOT NULL;

In [ ]:
-- Verify: top 10 categories (English names are now available)
SELECT
    product_category_english,
    COUNT(*) AS product_count
FROM SHOPFLOW_STAGED.STG_PRODUCTS
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10;

---
## Cell 8 — Create STG_PAYMENTS

### What this table contains
~103,886 rows — one row per payment event per order. An order can have multiple rows if the customer used multiple payment methods (e.g. part voucher, part credit card).

### Transformations applied

**`payment_value` → FLOAT**

The financial amount paid. Most important column in this table. Cast to FLOAT for aggregation.

**`payment_sequential` → INTEGER**

Order of payments for a single order (1, 2, 3...). Cast to INTEGER for correct sorting.

**`payment_installments` → INTEGER**

Number of monthly instalments. 1 = paid in full. Common values in Brazil: 1, 2, 3, 6, 10, 12. Cast to INTEGER for arithmetic.

### Why this table matters for analytics

To get total revenue for an order:
```sql
SELECT order_id, SUM(payment_value) AS total_paid
FROM STG_PAYMENTS
GROUP BY order_id;
```
Note: `SUM(payment_value)` across all rows for an order gives the correct total regardless of how many payment methods were used.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_PAYMENTS AS
SELECT
    order_id,
    TRY_CAST(payment_sequential AS INTEGER)    AS payment_sequential,  -- 1,2,3... per order
    payment_type,                              -- credit_card/boleto/voucher/debit_card
    TRY_CAST(payment_installments AS INTEGER)  AS payment_installments, -- 1=paid in full
    TRY_CAST(payment_value AS FLOAT)           AS payment_value,        -- amount in BRL
    _loaded_at,
    _source_file
FROM RAW_PAYMENTS
WHERE order_id IS NOT NULL
  AND payment_type IS NOT NULL;

In [ ]:
-- Verify: payment type breakdown with average values
SELECT
    payment_type,
    COUNT(*)                              AS payment_count,
    ROUND(AVG(payment_value), 2)          AS avg_value_brl,
    ROUND(AVG(payment_installments), 1)   AS avg_installments
FROM SHOPFLOW_STAGED.STG_PAYMENTS
GROUP BY 1
ORDER BY 2 DESC;

---
## Cell 9 — Create STG_REVIEWS

### What this table contains
~99,224 rows — one per customer review. Many customers only leave a star score — `review_comment_title` and `review_comment_message` are NULL for a large portion of rows.

### Transformations applied

**`review_score` → INTEGER**

Values 1–5. Cast from VARCHAR to INTEGER so it can be averaged, ordered, and grouped correctly. Without this cast, `ORDER BY review_score` would sort alphabetically: `1, 2, 3, 4, 5` happens to be the same order, but `ORDER BY review_score DESC` would give `5, 4, 3, 2, 1` — wrong if the column is VARCHAR in some edge cases.

**Two timestamps cast**

`review_creation_date` = when the review request was sent to the customer.
`review_answer_timestamp` = when the customer actually submitted their review.

The gap between the two (`review_response_days`) shows how long customers take to respond.

**Comment NULLs left as-is**

NULL comments are expected — most customers don't write text. We do NOT replace NULL comments with empty strings. NULL means 'no comment left'. Empty string means 'comment submitted but empty'. These are different and should be stored differently.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_REVIEWS AS
SELECT
    review_id,
    order_id,
    TRY_CAST(review_score AS INTEGER)             AS review_score,  -- 1 to 5 stars
    review_comment_title,                          -- often NULL — left as-is intentionally
    review_comment_message,                        -- often NULL — left as-is intentionally
    TRY_TO_TIMESTAMP(review_creation_date)         AS review_created_ts,
    TRY_TO_TIMESTAMP(review_answer_timestamp)      AS review_answered_ts,

    -- How long (in days) the customer took to submit their review after being asked
    DATEDIFF(
        'day',
        TRY_TO_TIMESTAMP(review_creation_date),
        TRY_TO_TIMESTAMP(review_answer_timestamp)
    ) AS review_response_days,

    _loaded_at,
    _source_file
FROM RAW_REVIEWS
WHERE review_id IS NOT NULL
  AND order_id IS NOT NULL
  AND review_score IS NOT NULL;

In [ ]:
-- Verify: review score distribution with comment rate
SELECT
    review_score,
    COUNT(*)                                        AS review_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct,
    COUNT(review_comment_message)                   AS with_comment,
    ROUND(COUNT(review_comment_message) * 100.0
          / COUNT(*), 1)                            AS comment_rate_pct
FROM SHOPFLOW_STAGED.STG_REVIEWS
GROUP BY 1
ORDER BY 1;

---
## Cell 10 — Create STG_GEOLOCATION

### What this table contains
Maps Brazilian zip code prefixes to lat/lng coordinates. The largest RAW table (~1M rows) but STAGED will be much smaller after deduplication.

### The duplicate problem

RAW_GEOLOCATION has multiple rows per zip code prefix — slightly different coordinates for the same zip. If you join customers to geolocation on zip, one customer row would match multiple coordinate rows — causing row fan-out and inflated counts.

You confirmed this in Cell 2 — some zip codes have hundreds of duplicate entries.

### Fix: ROW_NUMBER + QUALIFY

```sql
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY geolocation_zip_code_prefix
    ORDER BY _loaded_at DESC
) = 1
```

- `PARTITION BY zip` — groups all rows with the same zip together
- `ORDER BY _loaded_at DESC` — ranks the most recently loaded row as #1
- `QUALIFY ... = 1` — keeps only rank 1 per group — one row per zip

**What is QUALIFY?**

`QUALIFY` is Snowflake-specific syntax that filters rows based on window function results — without needing a CTE or subquery wrapper:

```sql
-- Standard SQL (verbose):
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (...) AS rn FROM ...
)
SELECT * FROM ranked WHERE rn = 1;

-- Snowflake QUALIFY (clean):
SELECT * FROM ...
QUALIFY ROW_NUMBER() OVER (...) = 1;
```

> **💡 Concept Note: ROW_NUMBER vs RANK**
> 
> `ROW_NUMBER()` vs `RANK()` vs `DENSE_RANK()`:
> - `ROW_NUMBER()` — always unique sequential numbers (1,2,3,4,5). No ties.
> - `RANK()` — tied rows share a rank, next rank skips (1,2,2,4,5).
> - `DENSE_RANK()` — tied rows share a rank, next rank does not skip (1,2,2,3,4).
> 
> For deduplication, always use `ROW_NUMBER()` — it guarantees exactly one row gets rank 1.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_GEOLOCATION AS
SELECT
    geolocation_zip_code_prefix              AS zip_code_prefix,  -- join key to customers + sellers
    TRY_CAST(geolocation_lat AS FLOAT)       AS lat,              -- latitude
    TRY_CAST(geolocation_lng AS FLOAT)       AS lng,              -- longitude
    geolocation_city                         AS city,
    UPPER(geolocation_state)                 AS state,
    _loaded_at,
    _source_file

FROM RAW_GEOLOCATION
WHERE geolocation_zip_code_prefix IS NOT NULL

-- QUALIFY filters on the window function result — Snowflake-specific syntax
-- ROW_NUMBER partitions by zip and orders by load time descending
-- = 1 keeps only the most recently loaded row per zip prefix
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY geolocation_zip_code_prefix
    ORDER BY _loaded_at DESC
) = 1;

In [ ]:
-- Verify: confirm deduplication worked — every zip should appear exactly once
SELECT
    COUNT(*)                              AS total_rows_after_dedup,
    COUNT(DISTINCT zip_code_prefix)       AS unique_zip_codes,
    -- If these two numbers match, deduplication is perfect
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT zip_code_prefix)
        THEN 'PASS — one row per zip'
        ELSE 'FAIL — duplicates remain'
    END AS dedup_check
FROM SHOPFLOW_STAGED.STG_GEOLOCATION;

---
## Cell 11 — Create STG_CATEGORY_TRANSLATION

### What this table contains
71 rows — maps Portuguese category names to English. The simplest table in the project.

It was already used as a JOIN source in `STG_PRODUCTS`. This STAGED version exists so Phase 3 (ANALYTICS) can reference it directly if needed, without going back to RAW.

No type casting needed — both columns are text. Just filter NULLs and keep it clean.

In [ ]:
CREATE OR REPLACE TABLE SHOPFLOW_STAGED.STG_CATEGORY_TRANSLATION AS
SELECT
    product_category_name,
    product_category_name_english,
    _loaded_at,
    _source_file
FROM RAW_CATEGORY_TRANSLATION
WHERE product_category_name IS NOT NULL
  AND product_category_name_english IS NOT NULL;

In [ ]:
-- Quick look at the translations
SELECT *
FROM SHOPFLOW_STAGED.STG_CATEGORY_TRANSLATION
ORDER BY product_category_name
LIMIT 10;

---
## Cell 12 — Verify all 9 STAGED tables

Query `INFORMATION_SCHEMA.TABLES` to confirm all 9 STAGED tables exist with the correct row counts.

### Expected behaviour

- `STG_GEOLOCATION` should have far fewer rows than `RAW_GEOLOCATION` — deduplication collapsed ~1M rows to ~19k unique zip codes.
- `STG_PRODUCTS` should have the same count as `RAW_PRODUCTS` — LEFT JOIN does not drop rows.
- All other tables should be close to their RAW equivalents, minus any NULL-filtered rows.

> **🔍 Concept Note: Real-time Metadata**
> 
> `INFORMATION_SCHEMA.TABLES` is real-time — row counts are always current. No latency. This is different from `ACCOUNT_USAGE.TABLES` which has up to 3-hour latency but covers the entire account and retains history for a year.

In [ ]:
-- All 9 STAGED tables with row counts
SELECT
    table_name,
    row_count,
    TO_CHAR(created, 'YYYY-MM-DD HH24:MI') AS created_at
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'SHOPFLOW_STAGED'
  AND table_name LIKE 'STG_%'
ORDER BY table_name;

---
## Cell 13 — Confirm data types are correct

Use `INFORMATION_SCHEMA.COLUMNS` to confirm that all the type casts actually worked — that `order_purchase_ts` is really `TIMESTAMP_NTZ` and not still `TEXT`.

This is the definitive check — more reliable than just looking at sample data.

In [ ]:
-- Check data types of STG_ORDERS — confirm timestamps are TIMESTAMP_NTZ not TEXT
SELECT
    column_name,
    data_type,
    is_nullable
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'SHOPFLOW_STAGED'
  AND table_name   = 'STG_ORDERS'
ORDER BY ordinal_position;

In [ ]:
-- Check data types of STG_GEOLOCATION — confirm lat/lng are FLOAT not TEXT
SELECT
    column_name,
    data_type,
    is_nullable
FROM SHOPFLOW_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'SHOPFLOW_STAGED'
  AND table_name   = 'STG_GEOLOCATION'
ORDER BY ordinal_position;

---
## Cell 14 — Cross-table referential integrity checks

### What is referential integrity?

Every `order_id` in `STG_ORDER_ITEMS` should have a matching row in `STG_ORDERS`. Every `product_id` in `STG_ORDER_ITEMS` should exist in `STG_PRODUCTS`. If orphan IDs exist, joins in Phase 3 will produce NULL dimension values — silently wrong analytics.

Snowflake does **not enforce foreign keys** by default (they are defined but not enforced for performance reasons). You must check integrity yourself.

> **💡 Concept Note: Foreign Keys in Snowflake**
> 
> Snowflake supports `FOREIGN KEY` constraints in DDL but marks them as `NOVALIDATE` and `RELY` by default — they are informational only, not enforced at write time. The optimizer may use them for query planning.

In [ ]:
-- Check 1: order_items with no matching order
-- Result should be 0 — every item must belong to a known order
SELECT COUNT(*) AS orphan_order_items
FROM SHOPFLOW_STAGED.STG_ORDER_ITEMS i
LEFT JOIN SHOPFLOW_STAGED.STG_ORDERS o
       ON i.order_id = o.order_id
WHERE o.order_id IS NULL;

In [ ]:
-- Check 2: order_items with no matching product
-- Result should be 0 — every item must reference a known product
SELECT COUNT(*) AS orphan_product_refs
FROM SHOPFLOW_STAGED.STG_ORDER_ITEMS i
LEFT JOIN SHOPFLOW_STAGED.STG_PRODUCTS p
       ON i.product_id = p.product_id
WHERE p.product_id IS NULL;

In [ ]:
-- Check 3: orders with no matching customer
-- Small number expected — Olist dataset has some known orphans
SELECT COUNT(*) AS orders_without_customer
FROM SHOPFLOW_STAGED.STG_ORDERS o
LEFT JOIN SHOPFLOW_STAGED.STG_CUSTOMERS c
       ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

---
## Cell 15 — Business sense check

Run a few end-to-end queries that join multiple STAGED tables together. If these produce sensible results, the STAGED layer is working correctly end to end.

These are also a preview of the kinds of questions Phase 3 (ANALYTICS) will answer — but from STAGED tables directly, before the star schema is built.

In [ ]:
-- Business check 1: Total GMV by order status
-- Delivered orders should dominate revenue
SELECT
    o.order_status,
    COUNT(DISTINCT o.order_id)          AS orders,
    ROUND(SUM(i.price + i.freight_value), 2) AS total_gmv_brl
FROM SHOPFLOW_STAGED.STG_ORDERS o
JOIN SHOPFLOW_STAGED.STG_ORDER_ITEMS i
  ON o.order_id = i.order_id
GROUP BY 1
ORDER BY 3 DESC;

In [ ]:
-- Business check 2: Average review score by product category
-- Shows which categories have happiest vs unhappiest customers
SELECT
    p.product_category_english,
    COUNT(DISTINCT r.review_id)         AS reviews,
    ROUND(AVG(r.review_score), 2)       AS avg_score
FROM SHOPFLOW_STAGED.STG_REVIEWS r
JOIN SHOPFLOW_STAGED.STG_ORDER_ITEMS i
  ON r.order_id = i.order_id
JOIN SHOPFLOW_STAGED.STG_PRODUCTS p
  ON i.product_id = p.product_id
GROUP BY 1
HAVING COUNT(DISTINCT r.review_id) >= 100  -- only categories with enough reviews
ORDER BY 3 DESC
LIMIT 10;

In [ ]:
-- Business check 3: Delivery delay vs review score
-- Late deliveries should correlate with lower review scores
SELECT
    CASE
        WHEN o.delivery_delay_days < 0  THEN 'Early'
        WHEN o.delivery_delay_days = 0  THEN 'On time'
        WHEN o.delivery_delay_days <= 7 THEN 'Up to 1 week late'
        ELSE 'More than 1 week late'
    END                                      AS delivery_bucket,
    COUNT(DISTINCT o.order_id)               AS orders,
    ROUND(AVG(r.review_score), 2)            AS avg_review_score
FROM SHOPFLOW_STAGED.STG_ORDERS o
JOIN SHOPFLOW_STAGED.STG_REVIEWS r
  ON o.order_id = r.order_id
WHERE o.delivery_delay_days IS NOT NULL
GROUP BY 1
ORDER BY avg_review_score DESC;

---
## Cell 16 — Suspend warehouse

Always suspend the warehouse explicitly at the end of a session. `SHOPFLOW_WH` has `AUTO_SUSPEND=60` but manual suspension stops the billing clock immediately.

In [ ]:
ALTER WAREHOUSE SHOPFLOW_WH SUSPEND;

In [ ]:
SHOW WAREHOUSES LIKE 'SHOPFLOW%';

---
## 🛠 Self-Guided Exploration:

Now that your Staged tables are built, let's verify that the transformations applied exactly the way you expect. Write your own SQL queries from scratch to solve these challenges.

### Challenge 1 (Easy): The Null Handling Check
**Objective:** Verify that `COALESCE` worked correctly in your `STG_PRODUCTS` table.
**Task:** Write a simple query to count how many rows in `SHOPFLOW_STAGED.STG_PRODUCTS` were explicitly assigned the `'uncategorised'` label for `product_category_english`.

In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2 (Medium): The Qualify Check
**Objective:** Prove that `ROW_NUMBER() + QUALIFY` successfully deduplicated the geolocation data.
**Task:** Write an aggregated query against `STG_GEOLOCATION` to find exactly how many unique `zip_code_prefix` values exist where the state is exactly `'SP'` (São Paulo).

In [ ]:
-- Write your SQL for Challenge 2 here


### Challenge 3 (Hard): The Delivered Revenue Check
**Objective:** Test your ability to join multiple staged tables safely.
**Task:** Write a query that joins `STG_ORDERS` and `STG_PAYMENTS` to calculate the total overall sum of `payment_value` specifically for orders that have an `order_status` of `'delivered'`.

In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4 (Complex): The SLA Investigator
**Objective:** Combine calculated fields, joins, and aggregates to answer a complex business question.
**Task:** Using `STG_ORDERS` and `STG_ORDER_ITEMS`, write a query to find the top 5 `seller_id`s who have the highest average `delivery_delay_days`. To make it statistically relevant, *only* include sellers who have processed at least 50 delivered orders.

In [ ]:
-- Write your SQL for Challenge 4 here


---
## Phase 2 complete

If Cell 12 shows all 9 STG tables, Cell 13 confirms proper data types, and Cell 14 shows 0 orphan records — your STAGED layer is ready.

### What you just built

```text
SHOPFLOW_STAGED
├── STG_ORDERS              ← timestamps cast · delivery_delay_days computed
├── STG_ORDER_ITEMS         ← price + freight as FLOAT
├── STG_CUSTOMERS           ← state normalised
├── STG_SELLERS             ← state normalised
├── STG_PRODUCTS            ← typos fixed · English category added
├── STG_PAYMENTS            ← payment_value as FLOAT
├── STG_REVIEWS             ← score as INTEGER · response_days computed
├── STG_GEOLOCATION         ← deduplicated · lat/lng as FLOAT
└── STG_CATEGORY_TRANSLATION ← clean lookup
```

### Key Snowflake concepts you practised

| Topic | Covered |
|-------|--------|
| TRY_CAST vs CAST | ✅ |
| TRY_TO_TIMESTAMP | ✅ |
| TIMESTAMP_NTZ | ✅ |
| CTAS pattern | ✅ |
| ROW_NUMBER + QUALIFY | ✅ |
| COALESCE | ✅ |
| LEFT JOIN vs INNER JOIN | ✅ |
| DATEDIFF | ✅ |
| INFORMATION_SCHEMA.COLUMNS | ✅ |
| Foreign key behaviour in Snowflake | ✅ |

### Next — Phase 3: ANALYTICS star schema

- Build `FACT_ORDERS` — one row per order item, all measures
- Build `DIM_CUSTOMERS`, `DIM_SELLERS`, `DIM_PRODUCTS`, `DIM_DATE`
- Write the business queries the project was built for
- Clustering keys for query performance